## Setup

This demo uses `matplotlib` (plotting) and `drawdata` (the interactive scatter widget), which are **not** part of the core `flash-ansr` install. Run the cell below once to install them.

In [ ]:
%pip install matplotlib drawdata

In [ ]:
from flash_ansr import FlashANSR, SoftmaxSamplingConfig

import torch
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [2]:
from drawdata import ScatterWidget

In [ ]:
# A checkpoint directory, e.g. after `flash_ansr install psaegert/flash-ansr-v25.0-T7-3M`
from flash_ansr import get_path
CHECKPOINT = get_path("models", "psaegert/flash-ansr-v25.0-T7-3M")

In [ ]:
nsr = FlashANSR.load(
    directory=CHECKPOINT,
    generation_config=SoftmaxSamplingConfig(choices=4096),  # more choices: a more thorough search
    ranking_mode="mdl",  # log10(FVU) + 1e-2 per bit of description length (default)
).to(device)

In [7]:
widget = ScatterWidget()
widget

In [ ]:
X_raw, y_raw = widget.data_as_X_y

# Scale X_raw and y_raw to (-10, 10)
X = (X_raw - X_raw.min(axis=0)) / (X_raw.max(axis=0) - X_raw.min(axis=0)) * 20 - 10
y = (y_raw - y_raw.min(axis=0)) / (y_raw.max(axis=0) - y_raw.min(axis=0)) * 20 - 10

In [ ]:
nsr.fit(X, y, verbose=True)

In [ ]:
nsr.get_expression()

In [ ]:
# Optional: re-rank the same candidates under a different rule without refitting
nsr.compile_results(ranking_mode="weighted", ranking_weights={"n_nodes": 0.05})

In [ ]:
nsr.results.head(10)

In [ ]:
X_linspace = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)

In [ ]:
# Predictions of the 32 best candidates on the data and on a dense grid
n_show = min(32, len(nsr.results))
y_pred = [nsr.predict(X, nth_best_beam=i) for i in range(n_show)]
y_pred_linspace = [nsr.predict(X_linspace, nth_best_beam=i) for i in range(n_show)]

In [ ]:
COLS = 8
ROWS = int(np.ceil(n_show / COLS))

fig, axs = plt.subplots(ROWS, COLS, figsize=(COLS * 4, ROWS * 4), dpi=150, sharex=True)

for i, ax in enumerate(np.atleast_1d(axs).flat):
    if i >= n_show:
        ax.axis('off')
        continue
    fvu = nsr.results.iloc[i]['fvu']
    ax.scatter(X[:, 0], y, s=4, color='black', label='data')
    ax.plot(X_linspace[:, 0], y_pred_linspace[i], color='tab:red', label='prediction')
    ax.set_title(f"#{i}: {nsr.get_expression(nth_best_beam=i, precision=3)}\nFVU = {fvu:.3g}", fontsize=8)
    ax.set_ylim(y.min() - 1, y.max() + 1)

plt.tight_layout()
plt.show()